In [1]:
import json
import re
import numpy as np
import os
import sys
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import StaticEmbedding
from postprocessing import postprocessing
import math

c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASUS\.conda\envs\env_py39\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_lg' (3.4.1) was trained with spaCy v3.4.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [2]:
DISTANCE_THRESHOLD = 0.12

In [3]:
def load_json(path):
    if not os.path.exists(path):
        print(f"Lỗi: Không tìm thấy file tại {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
    

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)
        
        
def extract_references(long_answer):
    if not long_answer:
        return []
    return re.findall(r"\[([^\]]+)\]", long_answer)

def cluster_cohesion(embeddings):
    if len(embeddings) <= 1:
        return 0
    centroid = embeddings.mean(axis=0)
    dists = np.linalg.norm(embeddings - centroid, axis=1)
    return dists.mean()

In [ ]:
def collect_candidates(cluster_file, mapping_data):
    raw_data = load_json(cluster_file)
    processed_data = {}

    for qidx, clusters in raw_data.items():
        cand_list = []
        for cid, cluster in clusters.items():
            if "llm" not in cluster:
                continue

            ans = cluster["llm"].get("short_answer", "").strip().lower()
            long_answer = cluster["llm"].get("long_answer", "")
            context = cluster.get("context", "").lower()
            
            if not ans or ans == "insufficient information":
                continue

            if ans in mapping_data:
                norm_ans = mapping_data[ans]
            else:
                norm_ans = postprocessing(ans)
                norm_ans = mapping_data.get(norm_ans, norm_ans)
                        
            raw_sources = extract_references(long_answer)
            sources = list(set(raw_sources))
   
            cand_list.append({
                "cid": cid,
                "text": norm_ans,
                "long_answer": long_answer,
                "sources": sources,
            })
        
        processed_data[qidx] = cand_list if len(cand_list) >= 1 else None
        
    return processed_data

In [ ]:
def select_best_cluster_via_llm(question_text, raw_clusters, cand_list):
    """
    Gửi toàn bộ candidate của tất cả các cụm cho LLM để chọn ra cụm tốt nhất.
    """
    if not raw_clusters:
        return None, "No clusters"

    # Chuẩn bị dữ liệu chi tiết từng cụm
    clusters_full_info = []
    for i, cluster in enumerate(raw_clusters):
        # Lấy TOÀN BỘ text của các candidate trong cụm này
        all_candidates_in_cluster = [cand_list[idx]["text"] for idx in cluster["indices"]]
        
        clusters_full_info.append({
            "cluster_index": i,
            "candidate_count": cluster["evidence_count"],
            "total_sources_count": cluster["unique_source"],
            "all_answers": all_candidates_in_cluster # Đưa toàn bộ vào đây
        })

    prompt = f"""Bạn là một chuyên gia thẩm định dữ liệu y sinh.
Nhiệm vụ: Phân tích các cụm câu trả lời bên dưới và chọn ra cụm (cluster) có nội dung chính xác, đồng nhất và đáng tin cậy nhất cho câu hỏi.

Câu hỏi: "{question_text}"

Dữ liệu các cụm (mỗi cụm chứa danh sách tất cả các ứng viên):
{json.dumps(clusters_full_info, ensure_ascii=False, indent=2)}

Tiêu chí đánh giá:
1. Sự đồng thuận: Cụm có nhiều ứng viên cùng trả lời một nội dung nhất quán.
2. Độ tin cậy: Tổng số nguồn (total_sources_count) của cụm lớn.
3. Nội dung: Câu trả lời phải giải quyết trực tiếp câu hỏi y khoa, không chọn các cụm chứa câu trả lời rác hoặc "insufficient information".

Trả về JSON duy nhất theo định dạng: {{"best_cluster_index": <int>, "reason": "<giải thích lý do chọn cụm này>"}}"""

    try:
        # Gọi hàm call_llm thực tế của bạn
        response_str = request_api(prompt) 
        res_data = json.loads(response_str)
        
        idx = res_data.get("best_cluster_index")
        reason = res_data.get("reason", "No reason provided")
        
        if 0 <= idx < len(raw_clusters):
            selected_cluster = raw_clusters[idx]
            selected_cluster["llm_reason"] = reason
            return selected_cluster, "llm_decision"
    except Exception as e:
        print(f"Lỗi khi LLM chọn cụm: {e}")
        
    # Fallback: Chọn cụm có tổng số nguồn lớn nhất nếu LLM lỗi
    fallback = max(raw_clusters, key=lambda x: x["unique_source"])
    fallback["llm_reason"] = "Fallback to Max Sources (LLM Error)"
    return fallback, "fallback_max_source"

In [ ]:
def cluster_and_evaluate(processed_data, model, dict_golden):
    fused_results = {}
    list_id_miss = set()
    detailed_logs = {} # Để lưu cache lại toàn bộ quá trình chọn

    for qidx, cand_list in processed_data.items():
        if not cand_list:
            fused_results[qidx] = {"abstain": True}
            list_id_miss.add(qidx)
            continue
        
        # --- BƯỚC 1: PHÂN CỤM (Giữ logic AgglomerativeClustering của bạn) ---
        texts = [c["text"] for c in cand_list]
        embeddings = model.encode(texts)
        raw_clusters = []

        if len(cand_list) >= 2:        
            from sklearn.cluster import AgglomerativeClustering
            labels = AgglomerativeClustering(
                n_clusters=None,
                distance_threshold=0.12, # Dựa trên biến DISTANCE_THRESHOLD của bạn
                metric="cosine", linkage="average"
            ).fit(embeddings).labels_

            for lbl in set(labels):
                indices = [i for i, l in enumerate(labels) if l == lbl]
                # Tính tổng số source của cả cụm
                total_refs = sum([len(cand_list[i]["sources"]) for i in indices])
                
                raw_clusters.append({
                    "indices": indices,
                    "evidence_count": len(indices),
                    "unique_source": total_refs
                })
        else:
            # Chỉ có 1 candidate
            raw_clusters = [{"indices": [0], "evidence_count": 1, "unique_source": len(cand_list[0]["sources"])}]

        # --- BƯỚC 2: GỬI TẤT CẢ CANDIDATE TRONG CÁC CỤM VÀO LLM ĐỂ CHỌN ---
        question_text = dict_golden.get(qidx, {}).get("question", "N/A")
        best_cluster, method = select_best_cluster_via_llm(question_text, raw_clusters, cand_list)

        # --- BƯỚC 3: TRÍCH XUẤT FINAL ANSWER TỪ CỤM ĐÃ CHỌN ---
        if best_cluster:
            chosen_indices = best_cluster["indices"]
            # Trong cụm tốt nhất LLM đã chọn, lấy candidate có nhiều nguồn tham chiếu nhất
            best_cand_idx = max(chosen_indices, key=lambda idx: len(cand_list[idx]["sources"]))
            final_cand = cand_list[best_cand_idx]

            fused_results[qidx] = {
                "final_answer": final_cand["text"],
                "context": final_cand.get("context", ""),
                "abstain": False,
                "debug_info": {
                    "method": method,
                    "reason": best_cluster.get("llm_reason")
                }
            }

            # Cache lại để debug
            detailed_logs[qidx] = {
                "question": question_text,
                "all_clusters": raw_clusters,
                "selected_index": best_cluster.get("cluster_index"),
                "llm_reason": best_cluster.get("llm_reason"),
                "final_answer": final_cand["text"]
            }
        else:
            fused_results[qidx] = {"abstain": True}
            list_id_miss.add(qidx)

    # Lưu log chi tiết
    os.makedirs('cache_logs', exist_ok=True)
    save_json(detailed_logs, 'cache_logs/full_cluster_llm_logs.json')

    return fused_results, list(list_id_miss)

In [ ]:
def compute_selected_summary(fused_results, dict_golden):
    total_with_evidence = len(fused_results)
    selected, correct_selected, total_wrong = 0, 0, 0
    list_id_wrong = []

    for qidx, info in fused_results.items():
        if info.get("abstain", True):
            continue

        selected += 1
        pred = str(info.get("final_answer", "")).lower().strip()
        context = str(info.get("context", "")).lower()
        golden_list = [str(g).lower().strip() for g in dict_golden.get(qidx, {}).get("True", [])]

        if pred in golden_list:
            correct_selected += 1
        else:
            list_id_wrong.append(qidx)
            total_wrong += 1
            

    acc = correct_selected / selected if selected else 0.0
    coverage = selected / total_with_evidence if total_with_evidence else 0.0

    return {
        "total": total_with_evidence, "selected": selected, "correct": correct_selected,
        "acc": acc, "coverage": coverage, "wrong": total_wrong
    }, list_id_wrong

In [7]:
def load_pubmed_model():
    print("Loading embedding model (PubMedBERT)...")    
    return SentenceTransformer("pritamdeka/S-BioBert-snli-multinli-stsb",device="cpu")

In [ ]:
BASE_FOLDER = "src/cluster/final"
PATH_GOLDEN = "data/all_answer_golden_56.json"
PATH_MAPPING = 'data/mapping_answer_gol.json'

PATH_CLUSTER = r'src\cluster\final\WH_answer_kmean_merge_google_wiki_7_only_GG.json' #only gg

In [165]:
model = load_pubmed_model()
mapping_data = load_json(PATH_MAPPING)
dict_golden = load_json(PATH_GOLDEN)

Loading embedding model (PubMedBERT)...


In [ ]:
print("Step 1: Collecting candidates...")
processed_candidates_mapping = collect_candidates(PATH_CLUSTER, mapping_data)

Step 1: Collecting candidates...


In [ ]:
results, list_id_miss = cluster_and_evaluate(processed_candidates_mapping, model, dict_golden)


In [ ]:
#vtmthu MAPPING
print(len(list_id_miss))
save_json(list_id_miss, f"list_id_miss_ct.json")
stats, list_id_wrong = compute_selected_summary(results, dict_golden)
print(f"Selected Answers   : {stats['selected']} (Coverage: {stats['coverage']*100:.2f}%)")
print(f"Correct Answers    : {stats['correct']} (Accuracy: {stats['acc']*100:.2f}%)")


MIN_SCORE_THRESHOLD:  1.0
42
Selected Answers   : 430 (Coverage: 91.10%)
Correct Answers    : 395 (Accuracy: 91.86%)
